# Comparing Unfitted Methods on a Common Geometry

This notebook is a tutorial-style comparison of unfitted finite element methods applied to the same embedded boundary-value problem.
The main idea is to keep the geometry, mesh, and PDE fixed, and then change only the discretization strategy.

This makes the comparison meaningful: differences in behavior can be attributed to the unfitted method itself rather than to a change in the model problem.

The notebook is organized in two parts:

1. A shared setup: background mesh, embedded geometry, problem data, and continuous model.
2. Separate method blocks that discretize that same model in different ways.

This structure is intended to support direct comparison across multiple unfitted formulations while keeping the underlying problem fixed.


In [ ]:
using Gridap
using GridapEmbedded


## 1. Model Problem

Let $\Omega \subset \mathbb{R}^2$ denote the embedded circular domain.
The notebook considers the Poisson problem

$$
-\Delta u = f \quad \text{in } \Omega,
$$

with Dirichlet boundary condition

$$
u = u_D \quad \text{on } \Gamma_D,
$$

where in this example $f=10$ and $u_D=1$ are constants.

In strong form, this is a scalar elliptic boundary-value problem posed on a geometry that is not fitted by the background Cartesian mesh.
The purpose of the notebook is to approximate this same continuous problem with multiple unfitted methods, so the model stated here is the common reference for every method block below.

The corresponding weak form is: find $u \in V_D$ such that

$$
\int_{\Omega} \nabla u \cdot \nabla v\,\mathrm{d}x = \int_{\Omega} f v\,\mathrm{d}x
$$

for all test functions $v \in V_0$, where

$$
V_D = \{ w \in H^1(\Omega) : w = u_D \text{ on } \Gamma_D \},
\qquad
V_0 = \{ w \in H^1(\Omega) : w = 0 \text{ on } \Gamma_D \}.
$$

Since the mesh does not conform to $\Gamma_D$, the Dirichlet condition is imposed weakly in the discrete formulations through Nitsche-type terms.
The methods differ in how they obtain a stable unfitted discretization of this same weak problem: CutFEM adds ghost-penalty stabilization on cut-cell skeletons, whereas AgFEM modifies the discrete space through cell aggregation.


## 2. Background Mesh

All methods considered in this notebook start from the same unfitted background mesh.
We build a uniform Cartesian mesh on a square, and the actual computational domain is later extracted from it by cutting with an embedded circle.

The parameter `n` controls the number of cells in each coordinate direction. Changing `n` therefore refines the common discretization baseline for every method in the comparison.


In [ ]:
# Background Mesh
n = 100
partition = (n,n)
pmin = Point(-1.0,-1.0)
pmax = Point(1.0,1.0)
bgmodel = CartesianDiscreteModel(pmin,pmax,partition)


## 3. Geometry Definition

The embedded geometry is a circle.
This gives a simpler 2D benchmark in which the physical domain boundary is smooth but not aligned with the background Cartesian mesh.

This is precisely the regime where unfitted methods are useful, and it gives a common geometric benchmark for comparing how different formulations handle cut cells and embedded boundaries.


In [ ]:
# Defining the geometry
R = 0.5
geo8 = disk(R)


## 4. Problem Data

We prescribe a constant source term `f` in the bulk domain and a constant Dirichlet value `ud` on the embedded boundary.
These simple data are deliberate: they keep the notebook focused on the comparison of unfitted discretizations, rather than on complications coming from variable coefficients or elaborate boundary data.


In [ ]:
# Forcing data
ud = 1
f = 10


## 5. Method 1: CutFEM

This is the first method block in the comparison.
It keeps the shared model problem unchanged and introduces the CutFEM discretization on the common background mesh.

In `GridapEmbedded`, the first key object is the cut discretization

- `cutgeo = cut(bgmodel, geo8)`

which stores how the analytical geometry intersects the background mesh. From this object, different triangulations can be extracted depending on which part of the cut configuration is needed.

The calls

- `Ω`: the physical domain where the PDE is solved,
- `Γd`: the embedded boundary where Dirichlet conditions are imposed,
- `Γg`: the ghost skeleton used for ghost-penalty stabilization.

correspond to the following API constructions:

- `Triangulation(cutgeo, PHYSICAL)` builds the mesh integration domain associated with the physical region where the PDE is solved.
- `EmbeddedBoundary(cutgeo)` extracts the embedded circle boundary.
- `GhostSkeleton(cutgeo)` builds the interior skeleton used by CutFEM stabilization. Conceptually, this is the set of faces across which one penalizes jumps of derivatives in cut regions.

The calls `get_normal_vector(Γd)` and `get_normal_vector(Γg)` return geometric normal fields that can be used inside variational forms. In the notebook, `n_Γd` enters the Nitsche boundary terms, while `n_Γg` enters the ghost-penalty jump term.

The measures

- `dΩ = Measure(Ω, degree)`
- `dΓd = Measure(Γd, degree)`
- `dΓg = Measure(Γg, degree)`

tell `Gridap` where each integral is evaluated and with which quadrature order. This is one of the central ideas in the API: the weak form is written symbolically, but every integral is tied to a specific triangulation through a `Measure`.

For the finite element space, the code first defines the active region

- `Ω_act = Triangulation(cutgeo, ACTIVE)`

and then builds

- `ReferenceFE(lagrangian, Float64, order)` for the local polynomial basis,
- `TestFESpace(...)` for the discrete test space,
- `TrialFESpace(V)` for the matching trial space.

The active triangulation is larger than the purely physical region: it contains all background cells that participate in the unfitted discretization. This is why CutFEM needs additional stabilization when some of those cells are only very small intersections of the true domain.

The bilinear form `a(u,v)` is then written almost exactly as the mathematics suggests:

- the volume integral `∫(∇(v)⋅∇(u)) * dΩ` is the standard Poisson term,
- the boundary integral on `dΓd` is the symmetric Nitsche enforcement of the Dirichlet condition,
- the skeleton integral on `dΓg` is the ghost penalty.

Written explicitly, the CutFEM discrete problem is: find $u_h \in U_h$ such that

$$
a_h(u_h,v_h) = l_h(v_h) \qquad \forall v_h \in V_h,
$$

with

$$
\begin{aligned}
a_h(u_h,v_h)
&= \int_{\Omega} \nabla v_h \cdot \nabla u_h\,\mathrm{d}x \\
&\quad + \int_{\Gamma_D} \left( \frac{\gamma_d}{h} v_h u_h - v_h \, n_{\Gamma_D}\!\cdot\!\nabla u_h - (n_{\Gamma_D}\!\cdot\!\nabla v_h) u_h \right) \, \mathrm{d}s \\
&\quad + \int_{\Gamma_g} \gamma_g h \, [n_{\Gamma_g}\!\cdot\!\nabla v_h] \, [n_{\Gamma_g}\!\cdot\!\nabla u_h] \, \mathrm{d}s,
\end{aligned}
$$

and

$$
l_h(v_h)
= \int_{\Omega} f v_h\,\mathrm{d}x
+ \int_{\Gamma_D} \left( \frac{\gamma_d}{h} v_h u_D - (n_{\Gamma_D}\!\cdot\!\nabla v_h) u_D \right) \, \mathrm{d}s.
$$

The ghost-penalty term deserves special attention.
In the code,

- `jump(n_Γg⋅∇(v))`

is the jump of the normal derivative of the test function across faces in the ghost skeleton, and similarly for `u`.
The corresponding bilinear contribution

$$
\int_{\Gamma_g} \gamma_g h \,[n_{\Gamma_g}\cdot \nabla v]\,[n_{\Gamma_g}\cdot \nabla u] \, \mathrm{d}s
$$

penalizes discontinuities of normal gradients across cut-cell neighbors in the active mesh.
Although the discrete space is already `H^1`-conforming, very small cut portions can still lead to poor conditioning and loss of robustness. The ghost penalty counteracts this by extending control from the physical part of the mesh to nearby cut cells.

From the API point of view, this is why `GhostSkeleton(cutgeo)` is needed: it provides the face-based integration domain on which `jump(...)` terms are assembled.
Without `Γg` and `dΓg`, the weak form would only see the physical volume and embedded boundary, and the CutFEM-specific stabilization mechanism would be absent.

The linear form `l(v)` contains the source term and the right-hand side of the Nitsche boundary contribution.
Finally, `AffineFEOperator(a,l,U,V)` assembles the discrete linear system, and `solve(op)` computes the FE solution `uh`.

In the notebook comparison, CutFEM serves as the formulation that stabilizes the problem by adding extra terms directly to the weak form.


In [ ]:
# CutFEM implementation

# Cut the background model
cutgeo = cut(bgmodel,geo8)

# Setup integration meshes
Ω = Triangulation(cutgeo,PHYSICAL)
Γd = EmbeddedBoundary(cutgeo)
Γg = GhostSkeleton(cutgeo)

# Setup normal vectors
n_Γd = get_normal_vector(Γd)
n_Γg = get_normal_vector(Γg)

# Setup Lebesgue measures
order = 1
degree = 2*order
dΩ = Measure(Ω,degree)
dΓd = Measure(Γd,degree)
dΓg = Measure(Γg,degree)

# Setup FE space
Ω_act = Triangulation(cutgeo,ACTIVE)
V = TestFESpace(Ω_act,ReferenceFE(lagrangian,Float64,order),conformity=:H1)
U = TrialFESpace(V)

# Weak form
γd = 10.0
γg = 0.1
h = (pmax - pmin)[1] / partition[1]

a(u,v) =
∫( ∇(v)⋅∇(u) ) * dΩ +
∫( (γd/h)*v*u - v*(n_Γd⋅∇(u)) - (n_Γd⋅∇(v))*u ) * dΓd +
∫( (γg*h)*jump(n_Γg⋅∇(v))*jump(n_Γg⋅∇(u)) ) * dΓg

l(v) =
∫( v*f ) * dΩ +
∫( (γd/h)*v*ud - (n_Γd⋅∇(v))*ud ) * dΓd

# FE problem
op = AffineFEOperator(a,l,U,V)
uh_cut = solve(op)

# Post processing
writevtk(Ω,"postprocess/trian_O")
writevtk(Γd,"postprocess/trian_Gd",cellfields=["normal"=>n_Γd])
writevtk(Γg,"postprocess/trian_Gg")
writevtk(Triangulation(bgmodel),"postprocess/bgtrian")
writevtk(Ω,"postprocess/cutfem_solution",cellfields=["uh"=>uh_cut])


## 6. Method 2: AgFEM

This section implements the aggregated finite element method (AgFEM) for exactly the same geometry, mesh, and Poisson problem used in the CutFEM block.
The comparison is therefore method-to-method: the continuous problem stays fixed, while the discrete stabilization mechanism changes.

As in the CutFEM block, the starting point is the cut discretization `cutgeo = cut(bgmodel, geo8)`. The important difference is what is done with the active cells afterwards.

The calls

- `Triangulation(cutgeo, PHYSICAL)`
- `Triangulation(cutgeo, ACTIVE)`
- `EmbeddedBoundary(cutgeo)`

have the same interpretation as before: they define the physical integration region, the active discrete region, and the embedded boundary where the Dirichlet condition is imposed.

The key AgFEM idea is to stabilize the unfitted discretization by constraining degrees of freedom associated with problematic cut cells through aggregates.
Unlike CutFEM, AgFEM does not introduce a ghost-penalty term on a skeleton. Instead, stability is built into the discrete space by replacing the standard active FE space with an aggregated one.

The corresponding discrete AgFEM problem has the same Nitsche treatment of the Dirichlet boundary, but no ghost-penalty contribution. It reads: find $u_h^{ag} \in U_h^{ag}$ such that

$$
a_h^{ag}(u_h^{ag},v_h) = l_h^{ag}(v_h) \qquad \forall v_h \in V_h^{ag},
$$

with

$$
\begin{aligned}
a_h^{ag}(u_h^{ag},v_h)
&= \int_{\Omega} \nabla v_h \cdot \nabla u_h^{ag}\,\mathrm{d}x \\
&\quad + \int_{\Gamma_D} \left( \frac{\gamma_d}{h} v_h u_h^{ag} - v_h \, n_{\Gamma_D}\!\cdot\!\nabla u_h^{ag} - (n_{\Gamma_D}\!\cdot\!\nabla v_h) u_h^{ag} \right) \, \mathrm{d}s,
\end{aligned}
$$

and

$$
l_h^{ag}(v_h)
= \int_{\Omega} f v_h\,\mathrm{d}x
+ \int_{\Gamma_D} \left( \frac{\gamma_d}{h} v_h u_D - (n_{\Gamma_D}\!\cdot\!\nabla v_h) u_D \right) \, \mathrm{d}s.
$$

Compared with CutFEM, the formulas differ only by the absence of the ghost term. The stabilization mechanism is moved from the bilinear form into the construction of the aggregated space itself.

The standard space is first built through

- `reffe_ag = ReferenceFE(lagrangian, Float64, order_ag)`
- `Vstd_ag = TestFESpace(Ω_ag_act, reffe_ag, conformity=:H1)`

which is the unfitted counterpart of an ordinary `H1` conforming Lagrange FE space on the active cells. At this point the space still has the same small-cut-cell difficulties that CutFEM addresses with ghost penalties.

The AgFEM-specific step is

- `aggregates = aggregate(AggregateCutCellsByThreshold(1.0), cutgeo, geo8, IN)`

This `GridapEmbedded` call computes a cell-to-cell aggregation map. Intuitively, cut cells that are poorly suited for stable approximation are attached to better interior cells, producing aggregate groups.
The strategy `AggregateCutCellsByThreshold(1.0)` tells the library to aggregate all cut cells according to that threshold-based rule.

The line

- `V_ag = AgFEMSpace(Vstd_ag, aggregates)`

is the central API call of the method. It constructs a new FE space whose basis functions are algebraically constrained according to the aggregate information. In other words, stabilization is encoded in the space itself rather than through additional bilinear-form terms.

After that point, the rest of the `Gridap` workflow becomes very similar to the CutFEM case:

- `U_ag = TrialFESpace(V_ag)` defines the trial space,
- `Measure(...)` objects define the integration domains,
- `a_ag(u,v)` and `l_ag(v)` are written in the same symbolic variational syntax,
- `AffineFEOperator(a_ag, l_ag, U_ag, V_ag)` assembles the system,
- `solve(op_ag)` computes the discrete solution `uh_ag`.

Notice that the weak form contains only the volume term and the Nitsche boundary term. There is no ghost skeleton and no ghost penalty contribution. This is the main conceptual contrast with CutFEM: AgFEM changes the space, while CutFEM changes the bilinear form.

For tutorial purposes, the code below also exports aggregate information on the background mesh through `color_aggregates(aggregates, bgmodel)`. This makes it possible to inspect how the aggregation pattern differs from the CutFEM treatment of small cut cells.


In [ ]:
# AgFEM implementation

# Cut the background model
cutgeo = cut(bgmodel,geo8)

# Setup integration meshes
Ω_ag = Triangulation(cutgeo,PHYSICAL)
Ω_ag_act = Triangulation(cutgeo,ACTIVE)
Γd_ag = EmbeddedBoundary(cutgeo)

# Setup normal vectors
n_Γd_ag = get_normal_vector(Γd_ag)

# Setup Lebesgue measures
order_ag = 1
degree_ag = 2*order_ag
dΩ_ag = Measure(Ω_ag,degree_ag)
dΓd_ag = Measure(Γd_ag,degree_ag)

# Setup the standard active FE space
reffe_ag = ReferenceFE(lagrangian,Float64,order_ag)
Vstd_ag = TestFESpace(Ω_ag_act,reffe_ag,conformity=:H1)

# Aggregate cut cells following the AgFEM strategy
aggregates = aggregate(AggregateCutCellsByThreshold(1.0),cutgeo,geo8,IN)
V_ag = AgFEMSpace(Vstd_ag,aggregates)
U_ag = TrialFESpace(V_ag)

# Weak form
γd_ag = 10.0
h_ag = (pmax - pmin)[1] / partition[1]

a_ag(u,v) =
∫( ∇(v)⋅∇(u) ) * dΩ_ag +
∫( (γd_ag/h_ag)*v*u - v*(n_Γd_ag⋅∇(u)) - (n_Γd_ag⋅∇(v))*u ) * dΓd_ag

l_ag(v) =
∫( v*f ) * dΩ_ag +
∫( (γd_ag/h_ag)*v*ud - (n_Γd_ag⋅∇(v))*ud ) * dΓd_ag

# FE problem
op_ag = AffineFEOperator(a_ag,l_ag,U_ag,V_ag)
uh_ag = solve(op_ag)

# Post processing
colors_ag = color_aggregates(aggregates,bgmodel)
writevtk(Triangulation(bgmodel),"postprocess/agfem_aggregates",celldata=["aggregate"=>aggregates,"color"=>colors_ag])
writevtk(Ω_ag,"postprocess/agfem_solution",cellfields=["uh"=>uh_ag])


## 7. Method 3: Shifted Boundary Method

This section implements a tutorial-style shifted boundary method (SBM) variant for the same Poisson problem used above.
The source term and Dirichlet data remain unchanged; only the treatment of the circular embedded boundary differs.

The block is written around the standard SBM ingredients:

- a surrogate boundary `Γs_sbm` built from the background mesh,
- a closest-point map `M(x)` from the surrogate boundary to the true boundary,
- a shift vector `d_sbm(x) = M(x) - x`,
- a transferred boundary datum `u_D(M(x))`,
- a first-order transfer operator

$$
S_h(w) = w + d_{\mathrm{sbm}}\cdot \nabla w.
$$

As in the reusable `EmbeddedBenchmark.jl` code, the surrogate boundary is built from `Interior(...)` and `Interface(...)` so that the geometry handling is explicit and reproducible.

For this particular geometry, the closest-point projection can be written analytically.
Since the embedded boundary is the circle of radius `R`, each quadrature point on `Γs_sbm` is projected radially onto that circle.
This provides both the true boundary point `Xd_sbm` and the shift field `d_sbm` used by the transfer operator.

The variational form below uses a shifted Dirichlet Nitsche structure on the surrogate boundary:

$$
S_h(v_h), \qquad S_h(u_h), \qquad n_{\Gamma_s}\cdot\nabla u_h,
$$

so both the trial trace and the test trace are transferred to first order while the Nitsche fluxes use the surrogate-boundary normal.
The physical domain is the disk itself, and the Dirichlet boundary is replaced by its surrogate counterpart on the background mesh.
For constant `ud` this transfer is trivial, but the code below evaluates the boundary datum at `Xd_sbm` so that the same block also works for spatially varying Dirichlet data.


In [ ]:
# Shifted Boundary Method implementation

# Build the SBM surrogate computational domain directly from geo8 itself
cutgeo = cut(bgmodel,geo8)
Ω_sbm = Interior(cutgeo, IN)

# Surrogate boundary is the boundary of the same surrogate computational domain
Ω_sbm_outside = Interior(cutgeo,ACTIVE_OUT)
Γs_sbm = Interface(Ω_sbm_outside,Ω_sbm).⁻
n_Γs_sbm = get_normal_vector(Γs_sbm)

# Measures and FE space
order_sbm = 1
degree_sbm = 2*order_sbm
dΩ_sbm = Measure(Ω_sbm,degree_sbm)
dΓs_sbm = Measure(Γs_sbm,degree_sbm)

reffe_sbm = ReferenceFE(lagrangian,Float64,order_sbm)
V_sbm = TestFESpace(Ω_sbm,reffe_sbm,conformity=:H1)
U_sbm = TrialFESpace(V_sbm)

# Closest-point map M(x), shift vector d(x)=M(x)-x, true boundary normal,
# and transferred Dirichlet data u_D(M(x))
function build_sbm_distance_data_circle(Γs,R,ud; degree)
    QΓs = CellQuadrature(Γs,degree)
    pts_Γs = get_cell_points(QΓs).cell_phys_point
    z = zero(VectorValue{2,Float64})
    Xd = CellState(z,QΓs)
    d = CellState(z,QΓs)
    n_true = CellState(z,QΓs)
    ud_shift = CellState(0.0,QΓs)
    ρ_tol = sqrt(eps(Float64))

    for icell in 1:length(pts_Γs)
        for ipoint in 1:length(pts_Γs[icell])
            x = pts_Γs[icell][ipoint]
            r = sqrt(x[1]^2 + x[2]^2)
            r <= ρ_tol && error("SBM closest-point map is undefined at x = $x")
            Xd_pt = Point(R*x[1]/r,R*x[2]/r)
            n_pt = VectorValue(Xd_pt[1]/R,Xd_pt[2]/R)

            Xd.values[icell][ipoint] = Xd_pt
            d.values[icell][ipoint] = Xd_pt - x
            n_true.values[icell][ipoint] = n_pt
            ud_shift.values[icell][ipoint] = ud isa Number ? Float64(ud) : ud(Xd_pt)
        end
    end
    return (; QΓs, Xd, d, n_true, ud_shift)
end

dist_sbm = build_sbm_distance_data_circle(Γs_sbm,R,ud,degree=degree_sbm)
QΓs_sbm = dist_sbm.QΓs
Xd_sbm = dist_sbm.Xd
d_sbm = dist_sbm.d
n_true_sbm = dist_sbm.n_true
ud_sbm = dist_sbm.ud_shift

# First-order shifted Dirichlet Nitsche formulation
γd_sbm = 10.0
h_sbm = (pmax - pmin)[1] / partition[1]
S_sbm(w) = w + d_sbm⋅∇(w)

a_sbm(u,v) =
∫( ∇(v)⋅∇(u) ) * dΩ_sbm +
∫( (γd_sbm/h_sbm)*S_sbm(v)*S_sbm(u) - v*(n_Γs_sbm⋅∇(u)) - (n_Γs_sbm⋅∇(v))*S_sbm(u) ) * dΓs_sbm

l_sbm(v) =
∫( v*f ) * dΩ_sbm +
∫( (γd_sbm/h_sbm)*S_sbm(v)*ud_sbm - (n_Γs_sbm⋅∇(v))*ud_sbm ) * dΓs_sbm

# FE problem
op_sbm = AffineFEOperator(a_sbm,l_sbm,U_sbm,V_sbm)
uh_sbm = solve(op_sbm)

# Post processing
normal_true_cell_sbm = [sum(n_true_sbm.values[icell]) / length(n_true_sbm.values[icell]) for icell in eachindex(n_true_sbm.values)]
shift_cell_sbm = [sum(d_sbm.values[icell]) / length(d_sbm.values[icell]) for icell in eachindex(d_sbm.values)]
writevtk(Γs_sbm,"postprocess/trian_Gs_sbm",cellfields=["normal_sur"=>n_Γs_sbm],celldata=["normal_true"=>normal_true_cell_sbm,"shift"=>shift_cell_sbm])
writevtk(Ω_sbm,"postprocess/sbm_solution",cellfields=["uh"=>uh_sbm])

## 8. Comparison Notes and Post-processing

The `writevtk` calls export method-specific geometrical and solution data that can be inspected in ParaView.
These files make it possible to compare not only the computed solutions, but also the auxiliary structures introduced by each unfitted method, such as ghost skeletons or aggregates.

Because every method reuses the same mesh size, geometry, source term, and boundary data, the notebook supports direct side-by-side comparison.
Useful comparison points include accuracy, conditioning, method-specific stabilization, ease of implementation, and sensitivity to small cut cells.

A simple quantitative check is to compute pairwise norms of the difference between the discrete solutions.
The code cell below evaluates these differences in the common physical domain using the `L2` norm.


In [ ]:
# Pairwise L2 norm checks between the three solutions

cutgeo_cmp = cut(bgmodel,geo8)
Ω_cmp = Triangulation(cutgeo_cmp,IN)
degree_cmp = 2*max(order,order_ag,order_sbm)
dΩ_cmp = Measure(Ω_cmp,degree_cmp)

l2_diff(u1,u2) = sqrt(sum(∫( (u1-u2)*(u1-u2) ) * dΩ_cmp))

norm_cutfem_agfem = l2_diff(uh_cut,uh_ag)
norm_cutfem_sbm = l2_diff(uh_cut,uh_sbm)
norm_agfem_sbm = l2_diff(uh_ag,uh_sbm)

println("L2(uh_cut - uh_ag)   = ", norm_cutfem_agfem)
println("L2(uh_cut - uh_sbm)  = ", norm_cutfem_sbm)
println("L2(uh_ag - uh_sbm) = ", norm_agfem_sbm)
